# Native PyTorch tutorial 6

Nine 3,000-epoch fits: baseline, four L2 coefficients and four L1 coefficients. Regularization is an explicit differentiable loss, independent of optimizer weight decay. The reference superresolution input ordering is corrected to [t,x].

Run all cells in order. Full scientific workloads are the default; `NIF_SMOKE=1` is a separate development run and does not establish full convergence. The original TensorFlow and `_colab` notebooks are preserved as legacy references.

In [ ]:
# Run this cell first in a clean Colab Python 3.11–3.13 session.
import os, sys, subprocess, importlib.util
from pathlib import Path
# A local checkout is used directly for development; Colab installs the public pytorch branch.
repo = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "nif" / "model.py").exists()), None)
if repo is not None:
    sys.path.insert(0, str(repo))
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
        "torch==2.14.1", "numpy==2.1.3", "scipy==1.16.3", "matplotlib==3.10.0",
        "protobuf==7.36.2", "google-crc32c==1.9.0", "tfrecord==1.14.6",
        "git+https://github.com/pswpswpsw/nif.git@pytorch"])
import copy, json, time, math
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from matplotlib import pyplot as plt
import nif
# FULL is the default. Set NIF_SMOKE=1 before starting only for a short development check.
SMOKE = os.environ.get("NIF_SMOKE", "0") == "1"
device = torch.device(os.environ.get("NIF_DEVICE", "cuda" if torch.cuda.is_available() else "cpu"))
torch.manual_seed(42)
np.random.seed(42)
if device.type == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
print({"torch": torch.__version__, "device": str(device), "mode": "SMOKE" if SMOKE else "FULL"})

OUT = Path(os.environ.get("NIF_OUTPUT_DIR", "pytorch_outputs")) / "tutorial6"
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Explicit native loops replace compile/fit/callbacks. The loss reduction is per row,
# then the mean over the batch; area weights multiply each row's MSE (not weight-normalized).
def dense_tanh(inputs, outputs):
    # Preserve the original Dense default: Glorot uniform kernel and zero bias.
    layer = nn.Linear(inputs, outputs)
    nn.init.xavier_uniform_(layer.weight)
    nn.init.zeros_(layer.bias)
    return nn.Sequential(layer, nn.Tanh())

def epochs(full):
    return 2 if SMOKE else full

def tensor(array, on_device=device):
    return torch.as_tensor(array, dtype=torch.float32, device=on_device)

def loader(features, target, batch_size, weight=None):
    arrays = [tensor(features, "cpu"), tensor(target, "cpu")]
    if weight is not None:
        arrays.append(tensor(weight, "cpu").reshape(-1))
    return DataLoader(TensorDataset(*arrays), batch_size=batch_size, shuffle=True,
                      drop_last=False, pin_memory=device.type == "cuda")

def predict(model, features, batch_size=1024):
    model.eval()
    result = []
    with torch.no_grad():
        for xb in torch.as_tensor(features, dtype=torch.float32).split(batch_size):
            result.append(model(xb.to(next(model.parameters()).device)).detach().cpu())
    return torch.cat(result).numpy()

def centralize_gradients(model):
    # TF kernels are [input,output]; native Linear stores [output,input].
    # Centralize along the input axes, preserving the original mathematical operation.
    for module in model.modules():
        for name, parameter in module.named_parameters(recurse=False):
            if parameter.grad is not None and parameter.ndim > 1:
                axes = tuple(range(1, parameter.ndim)) if isinstance(module, nn.Linear) else tuple(range(parameter.ndim-1))
                parameter.grad.sub_(parameter.grad.mean(dim=axes, keepdim=True))

def save_checkpoint(model, optimizer, path, epoch):
    payload = {"model_state": {k: v.detach().cpu() for k,v in model.state_dict().items()},
               "optimizer_state": optimizer.state_dict(), "epoch": epoch,
               "framework": "pytorch", "smoke": SMOKE}
    if hasattr(model, "cfg_shape_net"):
        payload.update(model_class=type(model).__name__, cfg_shape_net=model.cfg_shape_net,
                       cfg_parameter_net=model.cfg_parameter_net, mixed_policy="float32")
    torch.save(payload, path)

def save_and_check(model, optimizer, name, example, factory=None):
    path = OUT / (name + ".pt")
    save_checkpoint(model, optimizer, path, -1)
    checkpoint = torch.load(path, map_location=device, weights_only=True)
    if factory is None:
        factory = lambda: type(model)(copy.deepcopy(model.cfg_shape_net),
                                       copy.deepcopy(model.cfg_parameter_net), "float32")
    restored = factory().to(device)
    restored.load_state_dict(checkpoint["model_state"])
    restored.eval(); model.eval()
    with torch.no_grad():
        expected, actual = model(tensor(example)), restored(tensor(example))
    torch.testing.assert_close(actual, expected, rtol=1e-5, atol=1e-6)
    print("Checkpoint predictions verified:", path)
    return restored

def train(model, batches, optimizer, nepoch, name, schedule=None,
          centralization=False, loss_function=None, before_epoch=None):
    history = []
    start = time.monotonic()
    for epoch in range(nepoch):
        model.train()
        if schedule is not None:
            for group in optimizer.param_groups:
                group["lr"] = schedule(epoch, group["lr"])
        if before_epoch is not None:
            before_epoch(epoch)
        summed, seen = {}, 0
        for batch in batches:
            xb, yb = (item.to(device) for item in batch[:2])
            optimizer.zero_grad(set_to_none=True)
            if loss_function is None:
                per_row = (model(xb) - yb).square().mean(dim=-1)
                if len(batch) == 3:
                    per_row = per_row * batch[2].to(device).reshape(-1)
                data_loss = per_row.mean()
                values = {"mse": data_loss}
            else:
                data_loss, values = loss_function(model, xb, yb)
            regularization = model.regularization_loss(xb) if hasattr(model, "regularization_loss") else xb.new_zeros(())
            loss = data_loss + regularization
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite loss in {name}, epoch {epoch}")
            loss.backward()
            if centralization:
                centralize_gradients(model)
            optimizer.step()
            for key, value in {**values, "regularization": regularization, "loss": loss}.items():
                summed[key] = summed.get(key, 0.0) + float(value.detach()) * len(xb)
            seen += len(xb)
        row = {"epoch": epoch, "rows": seen, **{key: value / seen for key,value in summed.items()}}
        history.append(row)
        if epoch % 100 == 0 or epoch == nepoch - 1:
            print(name, row, "elapsed_seconds", round(time.monotonic()-start, 2))
            (OUT / (name + "_history.json")).write_text(json.dumps(history, indent=2))
        if epoch % 1000 == 0 or epoch == nepoch - 1:
            save_checkpoint(model, optimizer, OUT / (name + "_latest.pt"), epoch)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.semilogy([row["loss"] for row in history]); ax.set(xlabel="epoch", ylabel="loss", title=name)
    fig.tight_layout(); fig.savefig(OUT / (name + "_loss.png")); plt.show(); plt.close(fig)
    return history

def wave_figure(truth, prediction, name):
    fig, axs = plt.subplots(1,3,figsize=(15,3))
    for ax, values, title in zip(axs, (truth,prediction,prediction-truth), ("true","pred","error")):
        im=ax.contourf(tt,xx,values.reshape(10,200),levels=50,cmap="seismic")
        fig.colorbar(im,ax=ax); ax.set(xlabel="t",ylabel="x",title=title)
    fig.tight_layout(); fig.savefig(OUT / (name+"_reconstruction.png")); plt.show(); plt.close(fig)

def write_metrics(values):
    print(json.dumps(values, indent=2))
    (OUT / "metrics.json").write_text(json.dumps({"mode": "smoke" if SMOKE else "full", **values}, indent=2))


# What you will learn

In this tutorial, you will learn how to add 
1. L1 Regularization on weights and bias of **ParameterNet**
2. L2 Regularization on weights and bias of **ParameterNet**

You will also learn regularization has to be set small for SIREN. Because if the regularization is moderately large, it will easily deviate the gradient dynamics out of the well-prepared initialization of SIREN. This results in poor training behavior. 

# Take home message:
- If you want, you can regularize the ParameterNet in NIF.
- But you need to set the coefficient so small, that it won't influence the weights initially. 
- L2/L1 Regularization on ParameterNet does NOT help the spurious wave if you increase the resolution of spatial coordinates, without any training label.

# Generate a toy data - but with higher frequency

In [ ]:
NT=10 # 20
NX=200

x = np.linspace(0,1,NX,endpoint=False)
t = np.linspace(0,100,NT,endpoint=False)

xx,tt=np.meshgrid(x,t)

omega = 400
c = 0.12/20
x0 = 0.2

u = np.exp(-1000*(xx-x0-c*tt)**2)*np.sin(omega*(xx-x0-c*tt))

# vis
plt.figure()
for i in range(NT):
    plt.plot(x,u[i,:],'-',label=str(i) + '-th time')

plt.xlabel('$x$',fontsize=25)
plt.ylabel('$u$',fontsize=25)


# vis iso
plt.figure(figsize=(4,4))
ax = plt.axes(projection='3d')
ax.plot_surface(xx,tt,u,cmap="rainbow", lw=2)#,rstride=1, cstride=1)
ax.view_init(57, -80)
ax.set_xlabel(r'$x$',fontsize=25)
ax.set_ylabel(r'$t$',fontsize=25)
ax.set_zlabel(r'$u$',fontsize=25)
# Keep all three axis labels inside the ordinary inline tight crop.
ax.set_box_aspect(None, zoom=0.75)
ax.xaxis.labelpad = ax.yaxis.labelpad = ax.zaxis.labelpad = 12


In [ ]:
# prepare testing data: finer normalized coordinates; ALWAYS [parameter, coordinate] = [t,x].
resolution=41 if SMOKE else 371
test_x=np.linspace(-1,1,resolution)
test_t=np.linspace(-1,1,resolution)
test_xx,test_tt=np.meshgrid(test_x,test_t)
test_data=np.column_stack([test_tt.ravel(),test_xx.ravel()])


# Now, we first train the data without any regularization

In [ ]:
cfg_shape_net = {"use_resblock": False, "connectivity": "full", "input_dim": 1,
                 "output_dim": 1, "units": 30, "nlayers": 2,
                 "weight_init_factor": 0.01, "omega_0": 30.0}
cfg_parameter_net = {"use_resblock": False, "input_dim": 1, "latent_dim": 1,
                     "units": 30, "nlayers": 2, "activation": "swish"}
nepoch,lr,batch_size = 3000,1e-3,512
results=[]


In [ ]:
from nif.demo import TravelingWaveHighFreq
tw = TravelingWaveHighFreq()
train_data = tw.data
train_dataset = loader(train_data[:, :2], train_data[:, 2:3], batch_size=512)
test_x_phys=test_xx*tw.std[1]+tw.mean[1]
test_t_phys=test_tt*tw.std[0]+tw.mean[0]
u_test=np.exp(-1000*(test_x_phys-x0-c*test_t_phys)**2)*np.sin(omega*(test_x_phys-x0-c*test_t_phys))
def run_regularization(kind=None, coefficient=0.0):
    config=copy.deepcopy(cfg_parameter_net)
    if kind is not None: config[kind+"_reg"]=coefficient
    name="no_reg" if kind is None else f"{kind}_{coefficient}"
    model=nif.NIFMultiScale(copy.deepcopy(cfg_shape_net),config,"float32").to(device)
    optimizer=nif.optimizers.ReferenceAdam(model.parameters(),lr=1e-3,eps=1e-7)
    history=train(model,train_dataset,optimizer,epochs(nepoch),name)
    model=save_and_check(model,optimizer,name,train_data[:64,:2])
    train_pred=predict(model,train_data[:,:2]).ravel()
    test_pred=predict(model,test_data).reshape(test_tt.shape)
    fig,axs=plt.subplots(1,3,figsize=(15,3.5),layout="constrained")
    # A common normalized-u scale makes nearly constant predictions look constant.
    levels=np.linspace(min(train_data[:,2].min(),train_pred.min(),test_pred.min()),
                       max(train_data[:,2].max(),train_pred.max(),test_pred.max()),51)
    for ax,values,title in zip(axs[:2],(train_data[:,2],train_pred),("true","pred")):
        im=ax.contourf(tt,xx,values.reshape(10,200),levels=levels,cmap="seismic");ax.set_title(title)
    im=axs[2].contourf(test_tt,test_xx,test_pred,levels=levels,cmap="seismic");axs[2].set_title("super-resolution")
    fig.colorbar(im,ax=axs,label="normalized u")
    fig.suptitle(name);fig.savefig(OUT/(name+"_superresolution.png"));plt.show();plt.close(fig)
    physical_prediction=test_pred*tw.std[2]+tw.mean[2]
    result={"kind":kind,"coefficient":coefficient,"epochs":epochs(nepoch),
            "training_mse":float(np.mean((train_pred-train_data[:,2])**2)),
            "physical_superresolution_mse":float(np.mean((physical_prediction-u_test)**2)),
            "regularization_loss":float(model.regularization_loss(tensor(train_data[:64,:2])).detach())}
    results.append(result)
    return model
baseline=run_regularization()


# Let's start with L2 regularization.

## 1. We try a tiny little L2 regularization (1e-5) on ParameterNet 

In [ ]:
model_l2_1e5=run_regularization("l2",1e-5)


## 2. The result looks worse but it looks okay. Now we try 1e-4 regularization..

In [ ]:
model_l2_1e4=run_regularization("l2",1e-4)


## 3. Next 1e-3

In [ ]:
model_l2_1e3=run_regularization("l2",1e-3)


## Next 1e-2. 

In [ ]:
model_l2_1e2=run_regularization("l2",1e-2)


# Let's try L1 regularization instead 

## 1e-5

In [ ]:
model_l1_1e5=run_regularization("l1",1e-5)


## 1e-4

In [ ]:
model_l1_1e4=run_regularization("l1",1e-4)


## 1e-3

In [ ]:
model_l1_1e3=run_regularization("l1",1e-3)


## 1e-2

In [ ]:
model_l1_1e2=run_regularization("l1",1e-2)


In [ ]:
assert len(results)==9
write_metrics({"fits":9,"epochs_per_fit":epochs(nepoch),"resolution":resolution,"results":results})


# Summary

What we see:
1. In terms of spatial interpolation for high-frequency signal, NIF-Multiscale (with SIREN) produces spurious wiggles. 
  - However, you could still perform super-resolution in a fully supervised manner, i.e., just ask the NIF to memorize the super-resoluted label. But don't expect NIF with Multiscale to produce high frequency content for FREE! 
2. L2 or L1 regularization on ParameterNet does NOT help.
3. L2 or L1 regularization can lead to worse performance if it is not super small.